#### Import Library


In [48]:
# NOTE: json = read JSON
# NOTE: Path = safer file paths than raw strings
# NOTE: pandas = convert to table + export CSV

import json
from pathlib import Path
import pandas as pd


#### File path + existence check

In [49]:
# NOTE: Put the JSON file in the same folder as your notebook,
# or change the path here.

input_path = Path("medium_orders.json")
input_path

WindowsPath('medium_orders.json')

In [50]:
# NOTE: Always check the file exists before reading, saves time.
print("Exists?", input_path.exists)
print("Absolute path:", input_path.resolve())

Exists? <bound method Path.exists of WindowsPath('medium_orders.json')>
Absolute path: C:\Users\pdinh\Python\Python_Full_Course\Python Practice\JSON-Python\json-practice-medium\medium_orders.json


#### Load JSON + inspect the “shape”

In [51]:
# NOTE: Read the file text then parse JSON into Python objects (dict/list).
order = json.loads(input_path.read_text(encoding="utf-8"))
order

{'meta': {'source': 'demo_api', 'pulled_at': '2026-02-05T00:00:00Z'},
 'data': [{'order_id': 'ord_101',
   'order_time': '2026-02-04T12:01:00Z',
   'customer': {'customer_id': '201',
    'email': 'alice@example.com',
    'country': 'AU'},
   'items': [{'sku': 'A1', 'qty': 2, 'unit_price': '4.50'},
    {'sku': 'B2', 'qty': '1', 'unit_price': 10}],
   'payment': {'method': 'card', 'currency': 'AUD'}},
  {'order_id': 'ord_102',
   'order_time': 'bad_timestamp',
   'customer': {'customer_id': 202,
    'email': 'bob@example.com',
    'country': None},
   'items': [],
   'payment': {'method': 'paypal', 'currency': 'AUD'}}]}

In [52]:
# NOTE: Confirm top level structures
print("Top-level keys: ", list(order.keys()))

# NOTE: meta is usually infor about the API pull; not rows:
print("Meta: ", order.get("meta",None))

# NOTE: data is where the rows usually are (list of dict records)
records = order.get("data")
print("Type of data", type(records))
print("Length of data: ", len(records))

# NOTE: always print one record to understand columns and type
print("First records ", records[0] if records else None)

Top-level keys:  ['meta', 'data']
Meta:  {'source': 'demo_api', 'pulled_at': '2026-02-05T00:00:00Z'}
Type of data <class 'list'>
Length of data:  2
First records  {'order_id': 'ord_101', 'order_time': '2026-02-04T12:01:00Z', 'customer': {'customer_id': '201', 'email': 'alice@example.com', 'country': 'AU'}, 'items': [{'sku': 'A1', 'qty': 2, 'unit_price': '4.50'}, {'sku': 'B2', 'qty': '1', 'unit_price': 10}], 'payment': {'method': 'card', 'currency': 'AUD'}}


##### From our data, item is a list not a dict. So we will create 2 tables:
- orders_df (1row/order)
- items_df (1 row/ item)

In [53]:
# =========================
# CELL 2 — Build "orders" table (1 row per order) by FLATTENING nested JSON
# =========================

# NOTE: "data" is the list of orders (each order is a dict)
records = order.get("data",None)

order_rows = []

for r in records:
    # NOTE: customer/payment are nested dictis; use 'or {}' to avoid None errors
    customer = r.get("customer") or {}
    payment = r.get("payment") or {}

    # NOTE: orders table = 1 row per order (NO items columns here)
    order_rows.append(
        {
            "order_id": r.get("order_id"),
            "order_time": r.get("order_time"),
            "customer_id": customer.get("customer_id"),
            "email": customer.get("email"),
            "country": customer.get("country"),
            "payment_method": payment.get("method"),
            "currency": payment.get("currency")
        }
    )
print(order_rows)

# NOTE: put it in a data frame
order_df = pd.DataFrame(order_rows)
order_df

[{'order_id': 'ord_101', 'order_time': '2026-02-04T12:01:00Z', 'customer_id': '201', 'email': 'alice@example.com', 'country': 'AU', 'payment_method': 'card', 'currency': 'AUD'}, {'order_id': 'ord_102', 'order_time': 'bad_timestamp', 'customer_id': 202, 'email': 'bob@example.com', 'country': None, 'payment_method': 'paypal', 'currency': 'AUD'}]


,order_id,order_time,customer_id,email,country,payment_method,currency
0,ord_101,2026-02-04T12:01:00Z,201,alice@example.com,AU,card,AUD
1,ord_102,bad_timestamp,202,bob@example.com,NaN,paypal,AUD


In [58]:
# =========================
# CELL 3 — Build "order_items" table (1 row per item) by EXPLODING items[]
# =========================

items_rows = []

for r in records:
    order_id = r.get("order_id")
    order_time = r.get("order_time")
    currency = (r.get("payment") or {}).get("currency")

    # NOTE: items is a LIST; is missing or empty -) []
    items = r.get("items") or {}

    # NOTE: explode: create 1 row for each item in the list
    for it in items:
        items_rows.append({
            "order_id": order_id,
            "order_time": order_time,
            "currency": currency,
            "sku": it.get("sku"),
            "qty": it.get("qty"),
            "unit_price": it.get("unit_price"),
        })

print(items_rows)

# NOTE: put it to items_df
items_df = pd.DataFrame(items_rows)
items_df


[{'order_id': 'ord_101', 'order_time': '2026-02-04T12:01:00Z', 'currency': 'AUD', 'sku': 'A1', 'qty': 2, 'unit_price': '4.50'}, {'order_id': 'ord_101', 'order_time': '2026-02-04T12:01:00Z', 'currency': 'AUD', 'sku': 'B2', 'qty': '1', 'unit_price': 10}]


,order_id,order_time,currency,sku,qty,unit_price
0,ord_101,2026-02-04T12:01:00Z,AUD,A1,2,4.50
1,ord_101,2026-02-04T12:01:00Z,AUD,B2,1,10


#### Cleaning orders_df

In [55]:
order_df

,order_id,order_time,customer_id,email,country,payment_method,currency
0,ord_101,2026-02-04T12:01:00Z,201,alice@example.com,AU,card,AUD
1,ord_102,bad_timestamp,202,bob@example.com,NaN,paypal,AUD


In [56]:
order_df["order_time"] = pd.to_datetime(order_df["order_time"], utc=True, errors ="coerce")
# Put n/a for customer_id and convert to numeric value
order_df["customer_id"] = order_df["customer_id"].fillna(-1).astype("int64")
# Uppercase country safely
order_df["country"] = (order_df["country"].fillna("UNKNOWN")).astype(str).str.upper()
# Title-case payment_method safely
order_df["payment_method"] = order_df["payment_method"].astype(str).str.title()
# Uppercase currency safely
order_df["currency"] = (order_df["currency"]).astype(str).str.upper()

order_df


,order_id,order_time,customer_id,email,country,payment_method,currency
0,ord_101,2026-02-04 12:01:00+00:00,201,alice@example.com,AU,Card,AUD
1,ord_102,NaT,202,bob@example.com,UNKNOWN,Paypal,AUD


#### Cleaning items_df

In [57]:
items_df

,order_id,order_time,currency,sku,qty,unit_price
0,ord_101,2026-02-04T12:01:00Z,AUD,A1,2,4.50
1,ord_101,2026-02-04T12:01:00Z,AUD,B2,1,10


In [60]:
items_df["order_time"] = pd.to_datetime(items_df["order_time"], errors = "coerce")
items_df["currency"] = items_df["currency"].fillna("UNKNOW").astype(str).str.upper()
items_df["sku"] = items_df["sku"].fillna("UNKNOW").astype(str).str.upper()
items_df["qty"] = pd.to_numeric(items_df["qty"])
items_df["unit_price"] = pd.to_numeric(items_df["unit_price"], errors="coerce").fillna(0)
items_df["line_total"] = items_df["qty"] * items_df["unit_price"]

items_df


,order_id,order_time,currency,sku,qty,unit_price,line_total
0,ord_101,2026-02-04 12:01:00+00:00,AUD,A1,2,4.5,9.0
1,ord_101,2026-02-04 12:01:00+00:00,AUD,B2,1,10.0,10.0
